# Entrega 2: preparación de datos y baselines

**TP Integrador de Aprendizaje Automático**

Proyecto: rendimiento de cultivos por partido en la Provincia de Buenos Aires.

Este notebook ejecuta, desde cero:

1. La descarga y verificación del CSV oficial.
2. La limpieza y el registro de exclusiones.
3. El cálculo del promedio de las cinco campañas anteriores.
4. La construcción del rendimiento relativo y de la clase `campaña mala`.
5. El split temporal.
6. Los baselines `Dummy` y `promedio de 5 campañas`.

La fuente es pública y tiene licencia CC BY 4.0.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

from src.config import (
    CROPS,
    DATASET_MODELADO,
    FIGURAS_DIR,
    START_YEAR,
    END_YEAR,
    TEST_START,
    THRESHOLD,
    TRAIN_END,
    VALIDATION_END,
)
from src.descargar_datos import download
from src.preparar_datos import read_raw, clean_raw, build_datasets, class_summaries
from src.baselines import run_baselines

print("Raíz del proyecto:", ROOT)
print("Cultivos:", CROPS)


Raíz del proyecto: C:\Users\PC Blado\Desktop\ISFDyT57\Aprendizaje automatico\INTEGRADOR\proyecto
Cultivos: ['Maíz', 'Trigo total', 'Girasol', 'Soja total']


## 1. Fuente y descarga

El archivo se descarga del catálogo de Datos Abiertos de la Provincia de Buenos Aires. Si ya existe localmente, se reutiliza y se verifica su metadata.


In [2]:
raw_path = download()
print(raw_path)


Archivo existente: C:\Users\PC Blado\Desktop\ISFDyT57\Aprendizaje automatico\INTEGRADOR\proyecto\datos\raw\estimaciones-agricolas-1969_2026.csv
{
  "fuente": "https://catalogo.datos.gba.gob.ar/dataset/estimaciones-agro",
  "url_descarga": "https://catalogo.datos.gba.gob.ar/dataset/4ee50a8f-6866-4191-9bf8-aebb30b0d718/resource/d3e23031-164a-43e0-b104-01ced63e4abd/download/estimaciones-agricolas-1969_2026.csv",
  "licencia": "CC BY 4.0",
  "archivo": "estimaciones-agricolas-1969_2026.csv",
  "bytes": 2959265,
  "sha256": "a3b8e1295978343847324b2c3dff4d41b985f093372b34b1290cbc7ad75934e0",
  "filas": 53905,
  "columnas": [
    "cultivo",
    "campania",
    "municipio_id",
    "municipio_nombre",
    "superficie_sembrada",
    "superficie_cosechada",
    "produccion",
    "rendimiento"
  ],
  "descargado_utc": "2026-10-19T14:27:26+00:00"
}
C:\Users\PC Blado\Desktop\ISFDyT57\Aprendizaje automatico\INTEGRADOR\proyecto\datos\raw\estimaciones-agricolas-1969_2026.csv


## 2. Lectura y exploración inicial

La unidad de análisis es **partido × cultivo × campaña**. La variable numérica es el rendimiento en kg/ha.


In [3]:
raw = read_raw()
print("Filas:", len(raw))
print("Columnas:", list(raw.columns))
display(raw.head())
display(raw.dtypes.to_frame("tipo"))


Filas: 53905
Columnas: ['cultivo', 'campania', 'municipio_id', 'municipio_nombre', 'superficie_sembrada', 'superficie_cosechada', 'produccion', 'rendimiento']


,cultivo,campania,municipio_id,municipio_nombre,superficie_sembrada,superficie_cosechada,produccion,rendimiento
0,Ajo,1969/70,6854,25 De Mayo,3,3,10,3333
1,Ajo,1970/71,6854,25 De Mayo,1,1,3,3000
2,Ajo,1969/70,6014,Adolfo Gonzales Chaves,15,15,82,5467
3,Ajo,1970/71,6014,Adolfo Gonzales Chaves,10,10,55,5500
4,Ajo,1971/72,6014,Adolfo Gonzales Chaves,8,8,44,5500


,tipo
cultivo,str
campania,str
municipio_id,int64
municipio_nombre,str
superficie_sembrada,str
superficie_cosechada,str
produccion,str
rendimiento,str


## 3. Limpieza

Se conservan cuatro cultivos: maíz, trigo total, girasol y soja total. Se excluyen sorgo (alta proporción de rendimiento cero) y cebada cervecera (serie corta). También se excluye la campaña 2025/26 por estar incompleta.

Cada exclusión queda registrada en una tabla de auditoría.


In [4]:
limpio_base, registro = clean_raw(raw)
display(registro)
print("Filas después de la limpieza:", len(limpio_base))


,paso,filas_antes,filas_despues,filas_removidas,motivo
0,lectura_csv,53905,53905,0,Archivo original
1,filtrar_cultivos,53905,20495,33410,"Cultivos del alcance: Maíz, Trigo total, Giras..."
2,filtrar_campanias,20495,20393,102,Se conservan campañas 1969 a 2024; se excluye ...
3,quitar_municipio_ignorado,20393,20393,0,Municipio 'Ignorado' sin partido asignado
4,quitar_rendimiento_invalido,20393,20392,1,Rendimiento nulo o negativo
5,quitar_superficie_invalida,20392,20390,2,Superficie sembrada nula o no positiva
6,quitar_duplicados,20390,20390,0,"Duplicados por cultivo, campaña y municipio"
7,ordenar,20390,20390,0,"Orden por cultivo, municipio y campaña"


Filas después de la limpieza: 20390


## 4. Promedio de 5 campañas y objetivos

El promedio se calcula con una grilla calendario: para cada cultivo y municipio se usan las **cinco campañas inmediatamente anteriores**. Si falta un año, el promedio queda nulo y la fila no entra al modelado. Así se evita saltear huecos y se respeta el momento de la predicción.

El objetivo de regresión es:

`ratio = rendimiento / promedio_5_campañas_anteriores`

El objetivo de clasificación es:

`campaña_mala = 1 si ratio < 0,85`

El umbral 0,85 es provisorio y se analiza también con 0,80 y 0,90.


In [5]:
limpio, modelado = build_datasets(limpio_base)
print("Filas en el dataset limpio:", len(limpio))
print("Filas en el dataset de modelado:", len(modelado))
display(modelado.head())
display(modelado[["rendimiento", "prev5", "ratio", "mala85"]].describe().T)


Filas en el dataset limpio: 20390
Filas en el dataset de modelado: 15179


,cultivo,campania,municipio_id,municipio_nombre,superficie_sembrada,superficie_cosechada,produccion,rendimiento,campania_inicio,campania_fin,perdida_total,peso_superficie,prev5,ratio,mala80,mala85,mala90,split
0,Girasol,1981/82,6007,Adolfo Alsina,28000.0,26700.0,26000.0,974.0,1981,1982,False,28000.0,860.0,1.132558,0,0,0,train
1,Girasol,1982/83,6007,Adolfo Alsina,25000.0,24000.0,21600.0,900.0,1982,1983,False,25000.0,904.8,0.994695,0,0,0,train
2,Girasol,1983/84,6007,Adolfo Alsina,25000.0,24500.0,24500.0,1000.0,1983,1984,False,25000.0,934.8,1.069748,0,0,0,train
3,Girasol,1984/85,6007,Adolfo Alsina,28000.0,28000.0,32000.0,1143.0,1984,1985,False,28000.0,934.8,1.222721,0,0,0,train
4,Girasol,1985/86,6007,Adolfo Alsina,52000.0,50000.0,73800.0,1476.0,1985,1986,False,52000.0,1003.4,1.470999,0,0,0,train


,count,mean,std,min,25%,50%,75%,max
rendimiento,15179.0,3305.713749,2031.030958,0.0,1900.0,2664.0,4000.0,11800.0
prev5,15179.0,3180.297806,1886.887581,180.0,1879.8,2544.8,3835.1,10220.0
ratio,15179.0,1.060326,0.290847,0.0,0.905381,1.057402,1.208082,10.294118
mala85,15179.0,0.180842,0.3849,0.0,0.0,0.0,0.0,1.0


## 5. Balance de clases

La clase positiva es la campaña mala. Un accuracy alto puede esconder un recall nulo: si el 80% de las campañas son normales, un modelo que siempre dice "normal" acierta el 80% y no detecta nada.


In [6]:
by_crop, by_campaign = class_summaries(modelado)
display(by_crop)
display(by_campaign.tail(15))


,cultivo,split,n,positivos,tasa_positivos
0,Girasol,test,451,39,0.086475
1,Girasol,train,2862,450,0.157233
2,Girasol,validacion,413,55,0.133172
3,Maíz,test,606,143,0.235974
4,Maíz,train,3074,560,0.182173
5,Maíz,validacion,499,67,0.134269
6,Soja total,test,568,149,0.262324
7,Soja total,train,2266,380,0.167696
8,Soja total,validacion,451,77,0.170732
9,Trigo total,test,579,139,0.240069


,campania_inicio,split,n,positivos,tasa_positivos
29,2010,train,374,28,0.074866
30,2011,train,375,91,0.242667
31,2012,train,377,72,0.190981
32,2013,train,376,52,0.138298
33,2014,validacion,374,44,0.117647
34,2015,validacion,370,23,0.062162
35,2016,validacion,361,28,0.077562
36,2017,validacion,359,152,0.423398
37,2018,validacion,364,23,0.063187
38,2019,test,370,58,0.156757


In [7]:
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)

pivot = by_campaign.pivot(index="campania_inicio", columns="split", values="tasa_positivos")
ax = pivot.plot(kind="bar", figsize=(14, 4), width=0.85)
ax.set_title("Tasa de campañas malas por campaña (umbral 85%)")
ax.set_xlabel("Campaña (año de inicio)")
ax.set_ylabel("Tasa de positivos")
ax.legend(title="Split")
plt.tight_layout()
plt.savefig(FIGURAS_DIR / "balance_por_campania.png", dpi=150)
plt.show()


C:\Users\PC Blado\AppData\Local\Temp\ipykernel_15908\1017688664.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Split temporal

El test se reserva antes de elegir modelos o hiperparámetros:

| Split | Campañas |
|---|---|
| train | 1981/82 a 2013/14 |
| validación | 2014/15 a 2018/19 |
| test | 2019/20 a 2024/25 |

El split es por campaña, no al azar: todos los partidos de una misma campaña comparten el mismo shock climático.


In [8]:
print(f"Train: {START_YEAR} a {TRAIN_END}")
print(f"Validación: {TRAIN_END + 1} a {VALIDATION_END}")
print(f"Test: {TEST_START} a {END_YEAR}")
display(modelado.groupby(["split", "cultivo"]).size().unstack(fill_value=0))


Train: 1981 a 2013
Validación: 2014 a 2018
Test: 2019 a 2024


cultivo,Girasol,Maíz,Soja total,Trigo total
split,,,,
test,451,606,568,579
train,2862,3074,2266,2945
validacion,413,499,451,465


## 7. Baselines

Se comparan dos sistemas:

- **Dummy**: predice la media del ratio del train para cada cultivo. Es el baseline obligatorio de la consigna.
- **Promedio de 5 campañas**: predice `ratio = 1`, es decir, el rendimiento esperado igual al promedio de las cinco campañas anteriores. Es el baseline real del problema.

En clasificación, ambos predicen la clase mayoritaria y, por lo tanto, no generan alertas.


In [9]:
reg, clf, glob = run_baselines(verbose=False)
print("Regresión (test)")
display(reg[reg["split"] == "test"].round(3))
print("Clasificación (test)")
display(clf[clf["split"] == "test"].round(3))
print("Resumen global")
display(glob.round(3))


Regresión (test)


,split,cultivo,modelo,n,mae_ratio,rmse_ratio,r2_ratio,mae_kg,rmse_kg,r2_kg
8,test,Maíz,Dummy,606,0.195,0.275,-0.484,1370.842,1849.774,-0.089
9,test,Maíz,Promedio 5 campañas,606,0.153,0.236,-0.091,1042.169,1502.602,0.282
10,test,Trigo total,Dummy,579,0.196,0.268,-0.293,734.989,1044.152,-0.012
11,test,Trigo total,Promedio 5 campañas,579,0.179,0.240,-0.042,671.859,929.792,0.198
12,test,Girasol,Dummy,451,0.125,0.156,-0.170,279.031,346.974,0.425
13,test,Girasol,Promedio 5 campañas,451,0.112,0.145,-0.019,246.440,314.045,0.529
14,test,Soja total,Dummy,568,0.181,0.243,-0.296,499.275,691.785,-0.166
15,test,Soja total,Promedio 5 campañas,568,0.162,0.219,-0.054,437.690,601.927,0.117


Clasificación (test)


,split,cultivo,modelo,n,accuracy,precision,recall,f1,tp,fp,fn,tn,tasa_alertas
8,test,Maíz,Dummy,606,0.764,0.0,0.0,0.0,0,0,143,463,0.0
9,test,Maíz,Promedio 5 campañas,606,0.764,0.0,0.0,0.0,0,0,143,463,0.0
10,test,Trigo total,Dummy,579,0.760,0.0,0.0,0.0,0,0,139,440,0.0
11,test,Trigo total,Promedio 5 campañas,579,0.760,0.0,0.0,0.0,0,0,139,440,0.0
12,test,Girasol,Dummy,451,0.914,0.0,0.0,0.0,0,0,39,412,0.0
13,test,Girasol,Promedio 5 campañas,451,0.914,0.0,0.0,0.0,0,0,39,412,0.0
14,test,Soja total,Dummy,568,0.738,0.0,0.0,0.0,0,0,149,419,0.0
15,test,Soja total,Promedio 5 campañas,568,0.738,0.0,0.0,0.0,0,0,149,419,0.0


Resumen global


,split,modelo,n,mae_ratio,rmse_ratio,r2_ratio,mae_kg,rmse_kg,r2_kg,clf_n,clf_accuracy,clf_precision,clf_recall,clf_f1,clf_tp,clf_fp,clf_fn,clf_tn,clf_tasa_alertas
0,validacion,Dummy,1828,0.165,0.316,-0.000,620.290,936.049,0.847,1828,0.852,0.0,0.0,0.0,0,0,270,1558,0.0
1,validacion,Promedio 5 campañas,1828,0.176,0.323,-0.043,665.193,947.777,0.843,1828,0.852,0.0,0.0,0.0,0,0,270,1558,0.0
2,test,Dummy,2204,0.177,0.245,-0.313,755.772,1172.683,0.704,2204,0.787,0.0,0.0,0.0,0,0,470,1734,0.0
3,test,Promedio 5 campañas,2204,0.154,0.217,-0.035,626.276,980.542,0.793,2204,0.787,0.0,0.0,0.0,0,0,470,1734,0.0


In [10]:
test_reg = reg[reg["split"] == "test"].copy()
mae = test_reg.pivot(index="cultivo", columns="modelo", values="mae_ratio")
recall = clf[clf["split"] == "test"].pivot(index="cultivo", columns="modelo", values="recall")
display(mae.round(3))
display(recall.round(3))


modelo,Dummy,Promedio 5 campañas
cultivo,,
Girasol,0.125,0.112
Maíz,0.195,0.153
Soja total,0.181,0.162
Trigo total,0.196,0.179


modelo,Dummy,Promedio 5 campañas
cultivo,,
Girasol,0.0,0.0
Maíz,0.0,0.0
Soja total,0.0,0.0
Trigo total,0.0,0.0


In [11]:
ax = mae.plot(kind="bar", figsize=(10, 4), rot=0)
ax.set_title("MAE del ratio en test")
ax.set_ylabel("MAE (ratio)")
ax.set_xlabel("Cultivo")
ax.legend(title="Modelo")
plt.tight_layout()
plt.savefig(FIGURAS_DIR / "baselines_mae_ratio_test.png", dpi=150)
plt.show()


C:\Users\PC Blado\AppData\Local\Temp\ipykernel_15908\2425859642.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Lectura de los resultados

Los números que importan:

- El **promedio de 5 campañas** tiene un MAE de ratio cercano a 0,15 en test y supera al Dummy, que se queda con la media del train.
- En **validación** el Dummy gana por poco; en **test** pierde. El período de prueba es peor que el de entrenamiento, sobre todo por la sequía 2022/23.
- Ambos baselines tienen **recall = 0**. Aciertan aproximadamente el 79% de las filas y no detectan ninguna campaña mala. Es el ejemplo perfecto de "buen MAE, mala clasificación".
- La campaña **2022/23** concentra alrededor del 61% de los casos malos del test. El modelo de la Clase 3 tiene que demostrar que puede detectarla con clima.


In [12]:
test_2022 = by_campaign[by_campaign["campania_inicio"] == 2022]
display(test_2022)

print("MAE ratio del baseline Promedio 5 en test:",
      round(float(glob[(glob.split == "test") & (glob.modelo == "Promedio 5 campañas")]["mae_ratio"].iloc[0]), 4))
print("Recall del baseline Promedio 5 en test:",
      round(float(glob[(glob.split == "test") & (glob.modelo == "Promedio 5 campañas")]["clf_recall"].iloc[0]), 4))
print("Tasa de positivos en test:", round(float(modelado[modelado.split == "test"]["mala85"].mean()), 4))


,campania_inicio,split,n,positivos,tasa_positivos
41,2022,test,366,222,0.606557


MAE ratio del baseline Promedio 5 en test: 0.1538
Recall del baseline Promedio 5 en test: 0.0
Tasa de positivos en test: 0.2132


## 9. Clima

El dataset de modelado de esta entrega contiene los predictores agrícolas y el promedio histórico. La descarga de clima de NASA POWER está implementada en `src/clima.py` y se ejecuta una vez:

```bash
python -m src.clima --full
python -m src.clima --merge
```

El archivo `datos/procesado/clima_features.csv` agrega, por municipio, cultivo y campaña: precipitación, días con máxima mayor a 30 °C, temperatura media, radiación, humedad, sequía máxima, precipitación previa a la siembra, días de helada y días de lluvia. La unión se usa en la Clase 3.

## 10. Próximos pasos

1. Incorporar las variables climáticas al dataset de modelado.
2. Entrenar y comparar los regresores obligatorios: Dummy, DecisionTree y KNN con StandardScaler dentro de un Pipeline.
3. Evaluar MAE, RMSE y R², analizar residuos y errores por campaña.
4. Seleccionar el regresor con validación y reservar el test para el final.

## 11. Reproducibilidad

```bash
python -m src.descargar_datos
python -m src.preparar_datos
python -m src.baselines
jupyter nbconvert --to notebook --execute --inplace notebooks/02_preparacion_y_baselines.ipynb
```
